# Supervised Fine-Tuning for Code Bug Detection on Microsoft Foundry

Code review is critical, but it is time-consuming and subtle bugs slip through even experienced reviewers. In this notebook, we teach **GPT-4.1-mini** to identify and explain bugs using supervised fine-tuning (SFT).

### What You'll Learn

1. Evaluate a base deployment to establish a baseline.
2. Submit and monitor an SFT fine-tuning job.
3. Resolve and evaluate its fine-tuned deployment.
4. Compare quality, cost, and latency against the teacher model.

### SDK Pattern

The notebook authenticates with `DefaultAzureCredential`, creates `AIProjectClient` from a Microsoft Foundry project endpoint, and gets the official OpenAI child client with `get_openai_client()`. It does not use API keys, construct an OpenAI client directly, configure an account endpoint, or call raw service routes.

### Notebook Roadmap

| Step | What | Tool |
|------|------|------|
| 1 | Setup and configuration | Azure AI Projects SDK |
| 2 | Inspect training data | Python |
| 3 | Baseline evaluation | Official OpenAI child client |
| 4 | Upload data and submit training | Official OpenAI child client |
| 5 | Monitor training | Official OpenAI child client |
| 6 | Resolve fine-tuned deployment | Projects deployment child client |
| 7 | Evaluate and compare | Official OpenAI child client |


## 1. Setup & Configuration

### Prerequisites

1. A Microsoft Foundry project with fine-tuning enabled.
2. Base and judge deployments named `gpt-4.1-mini` and `gpt-5-4`.
3. A `DefaultAzureCredential` identity with Foundry User access.
4. `FOUNDRY_PROJECT_ENDPOINT` set in `.env`.
5. Optional: set `FINE_TUNE_JOB_ID` and `FINE_TUNED_DEPLOYMENT` to replay evaluation for a succeeded job and its existing project deployment without retraining.


In [1]:
import json
import os
import re
import time
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
EXISTING_FINE_TUNE_JOB_ID = os.environ.get("FINE_TUNE_JOB_ID")
EXISTING_FINE_TUNED_DEPLOYMENT = os.environ.get("FINE_TUNED_DEPLOYMENT")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=FOUNDRY_PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()

BASE_MODEL = "gpt-4.1-mini"
SFT_BASE_MODEL = "gpt-4.1-mini-2025-04-14"
JUDGE_MODEL = "gpt-5-4"

deployments = {deployment["name"]: deployment for deployment in project_client.deployments.list()}
for required_deployment in (BASE_MODEL, JUDGE_MODEL):
    if required_deployment not in deployments:
        raise RuntimeError(f"Required deployment is missing: {required_deployment}")

print(f"Project endpoint: {FOUNDRY_PROJECT_ENDPOINT}")
print(f"Base deployment: {BASE_MODEL}")
print(f"SFT base model: {SFT_BASE_MODEL}")
print(f"Judge deployment: {JUDGE_MODEL}")
print("Setup complete!")


Project endpoint: https://swec-prakharg-demo-2026.services.ai.azure.com/api/projects/swec-prakharg-demo-2026
Base deployment: gpt-4.1-mini
SFT base model: gpt-4.1-mini-2025-04-14
Judge deployment: gpt-5-4
Setup complete!


## 2. Load & Inspect Training Data

Good training data is the single biggest factor in fine-tuning success. Let's look at what we're working with.

### About the Dataset

- **224 training examples** + **20 validation examples** of buggy code paired with expert-quality analysis
- Each example covers a different bug across **10 categories** (off-by-one errors, null references, type mismatches, resource leaks, race conditions, buffer overflows, integer overflow, logic errors, unhandled exceptions, and security vulnerabilities) in **multiple languages** (Python, JavaScript, Java, C++)
- This is a **distillation dataset** — the training examples were generated by GPT-5.4 (the teacher model) and then quality-filtered. Only examples scoring **≥7/10** on a quality rubric were kept. This curation step is critical: in our experiments, 224 high-quality examples outperformed 1,576 noisy ones.

### The SFT Chat Format

Each training example is a JSON object with a `messages` array in the standard chat format:

| Role | Purpose |
|------|------|
| **system** | Instructions for the bug detection task — tells the model what kind of response to produce |
| **user** | A code snippet containing a subtle bug |
| **assistant** | The ideal response — identifies the bug, explains why it's wrong, and suggests a minimal fix |

During fine-tuning, the model learns to generate the **assistant** message given the system + user context. That's the core of SFT: learning by example.


In [2]:
DATA_DIR = os.path.join("..", "..", "Sample_Datasets", "Supervised_Fine_Tuning", "Text-Bug-Detection")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

training_data = load_jsonl(os.path.join(DATA_DIR, "training_data.jsonl"))
validation_data = load_jsonl(os.path.join(DATA_DIR, "validation_data.jsonl"))

print(f"Training examples: {len(training_data)}")
print(f"Validation examples: {len(validation_data)}")

# Show one example
example = training_data[0]
msgs = example["messages"]
print("\n--- Example ---")
print(f"System: {msgs[0]['content'][:120]}...")
print(f"User:   {msgs[1]['content'][:120]}...")
print(f"Asst:   {msgs[2]['content'][:120]}...")

Training examples: 224
Validation examples: 20

--- Example ---
System: Given a Python function with a subtle bug, identify the exact bug, explain why it causes incorrect behavior with a speci...
User:   def add_tag(posts, tag):
    posts_copy = posts.copy()
    for post in posts_copy:
        post['tags'].append(tag)
    ...
Asst:   **Bug**: posts.copy() makes a shallow copy of posts list, but 'tags' lists inside each post dict are still shared refere...


## 3. Baseline Evaluation

Before fine-tuning, we need to know how well the base model already performs. This is your **control group** — without it, you can't measure whether fine-tuning actually helped. (Sometimes the base model is already good enough, and fine-tuning would be wasted effort!)

### Evaluation Approach

1. **Run** each test example through the base GPT-4.1-mini model
2. **Score** each response using a judge model (GPT-5.4) on three dimensions:
   - **Bug identification** (1–10): Did the model correctly find the bug?
   - **Explanation quality** (1–10): Is the explanation clear, accurate, and complete?
   - **Fix quality** (1–10): Is the proposed fix correct and following best practices?
3. **Pass/fail**: A combined score ≥ 7.0 counts as a "pass"

We use the **same judge model and rubric** for every evaluation in this notebook — the baseline, the fine-tuned model, and the teacher. Consistency is key: without a fixed rubric, you can't compare results across runs.

The next two code cells define the helper functions (`get_completion`, `judge_response`, `evaluate_model`) and then run the baseline evaluation.


In [3]:
# Prepare test examples (use validation set, cap at 10 for cost efficiency)
test_examples = validation_data[:10]
print(f"Using {len(test_examples)} test examples for evaluation")


def get_completion(model, messages, max_completion_tokens=2048, response_format=None):
    """Get a chat completion from the given model with bounded retries."""
    kwargs = {
        "model": model,
        "messages": messages,
        "max_completion_tokens": max_completion_tokens,
    }
    if model.startswith("gpt-5"):
        kwargs["reasoning_effort"] = "low"
    if response_format is not None:
        kwargs["response_format"] = response_format

    for attempt in range(1, 4):
        try:
            response = openai_client.chat.completions.create(**kwargs)
            content = response.choices[0].message.content
            if not content:
                raise RuntimeError("Model returned an empty response")
            return content
        except Exception as exc:
            if attempt == 3:
                raise
            wait_seconds = 5 * attempt
            print(
                f"  Completion retry {attempt}/2 for {model}: "
                f"{type(exc).__name__}; waiting {wait_seconds}s"
            )
            time.sleep(wait_seconds)


def judge_response(user_input, model_response, reference_response):
    """Use the judge model to score a bug detection response."""
    rubric = (
        "You are an expert code reviewer judging bug detection responses.\n"
        "Score the candidate response compared to the reference on three criteria, each 1-10:\n"
        "- bug_identification: Did the candidate correctly identify the bug?\n"
        "- explanation: Is the explanation clear, accurate, and include a failing test case?\n"
        "- fix_quality: Is the proposed fix correct and minimal?\n\n"
        "Return ONLY a JSON object with these three keys and integer scores.\n"
        'Example: {"bug_identification": 8, "explanation": 7, "fix_quality": 9}'
    )

    prompt = (
        "## Buggy Code\n" + user_input + "\n\n"
        "## Reference Answer\n" + reference_response + "\n\n"
        "## Candidate Answer\n" + model_response + "\n\n"
        "Score the candidate answer. Return ONLY the JSON object."
    )

    judge_messages = [
        {"role": "system", "content": rubric},
        {"role": "user", "content": prompt},
    ]

    raw = get_completion(
        JUDGE_MODEL,
        judge_messages,
        max_completion_tokens=4096,
        response_format={"type": "json_object"},
    )
    return parse_judge_scores(raw)


def parse_judge_scores(raw):
    """Extract and validate JSON scores from the judge response."""
    cleaned = re.sub(r"```json\s*", "", raw)
    cleaned = re.sub(r"```\s*", "", cleaned).strip()
    scores = json.loads(cleaned)
    expected_keys = {"bug_identification", "explanation", "fix_quality"}
    if set(scores) != expected_keys:
        raise ValueError(f"Judge returned unexpected keys: {sorted(scores)}")
    for key, value in scores.items():
        if not isinstance(value, int) or not 1 <= value <= 10:
            raise ValueError(f"Judge returned invalid score for {key}: {value}")
    return scores


Using 10 test examples for evaluation


In [4]:
def evaluate_model(model_name, test_examples, label="Model"):
    """Run test examples through a model and judge the responses."""
    results = []
    for i, example in enumerate(test_examples):
        msgs = example["messages"]
        system_msg = msgs[0]["content"]
        user_msg = msgs[1]["content"]
        reference = msgs[2]["content"]

        # Get model response
        response = get_completion(
            model_name,
            [{"role": "system", "content": system_msg}, {"role": "user", "content": user_msg}],
        )
        time.sleep(0.5)

        # Judge the response
        scores = judge_response(user_msg, response, reference)
        time.sleep(0.5)

        combined = (
            scores.get("bug_identification", 0)
            + scores.get("explanation", 0)
            + scores.get("fix_quality", 0)
        ) / 3.0

        results.append({"scores": scores, "combined": combined})
        print(
            f"  [{label}] Example {i+1}: "
            f"bug={scores.get('bug_identification', 0)} "
            f"expl={scores.get('explanation', 0)} "
            f"fix={scores.get('fix_quality', 0)} "
            f"combined={combined:.1f}"
        )

    # Summary
    avg_combined = sum(r["combined"] for r in results) / len(results)
    pass_count = sum(1 for r in results if r["combined"] >= 7.0)
    pass_rate = pass_count / len(results) * 100

    print(f"\n  {label} Summary:")
    print(f"    Average combined score: {avg_combined:.2f}")
    print(f"    Pass rate (>=7.0): {pass_count}/{len(results)} ({pass_rate:.1f}%)")

    return {"results": results, "avg_combined": avg_combined, "pass_rate": pass_rate}


print("Evaluating base model (gpt-4.1-mini)...\n")
baseline_eval = evaluate_model(BASE_MODEL, test_examples, label="Baseline")

Evaluating base model (gpt-4.1-mini)...



  [Baseline] Example 1: bug=9 expl=7 fix=8 combined=8.0


  [Baseline] Example 2: bug=10 expl=10 fix=10 combined=10.0


  [Baseline] Example 3: bug=10 expl=10 fix=9 combined=9.7


  [Baseline] Example 4: bug=2 expl=4 fix=2 combined=2.7


...

  [Baseline] Example 6: bug=10 expl=10 fix=10 combined=10.0


  [Baseline] Example 7: bug=10 expl=10 fix=10 combined=10.0


  [Baseline] Example 8: bug=10 expl=10 fix=9 combined=9.7


  [Baseline] Example 9: bug=10 expl=9 fix=6 combined=8.3


  [Baseline] Example 10: bug=10 expl=10 fix=10 combined=10.0

  Baseline Summary:
    Average combined score: 8.57
    Pass rate (>=7.0): 9/10 (90.0%)


## 4. Upload Training Data

Azure AI Foundry needs your training data uploaded as files before you can reference them in a fine-tuning job. This is a two-step process:

1. **Upload** the JSONL files using the Files API
2. **Wait** for the platform to validate and process them

The platform validates the JSONL format during upload — if there are schema issues (missing `messages` field, wrong role names, encoding errors), you'll get errors here rather than during training. This is a good thing: it's much faster to fix a data issue than to wait for a training job to fail.


In [5]:
if EXISTING_FINE_TUNE_JOB_ID:
    print(f"Reusing fine-tuning job: {EXISTING_FINE_TUNE_JOB_ID}")
    train_file = None
    val_file = None
else:
    print("Uploading training file...")
    with open(os.path.join(DATA_DIR, "training_data.jsonl"), "rb") as f:
        train_file = openai_client.files.create(file=f, purpose="fine-tune")
    print(f"Training file ID: {train_file.id}")

    print("\nUploading validation file...")
    with open(os.path.join(DATA_DIR, "validation_data.jsonl"), "rb") as f:
        val_file = openai_client.files.create(file=f, purpose="fine-tune")
    print(f"Validation file ID: {val_file.id}")

    print("\nWaiting for files to be processed...")
    for file_id in [train_file.id, val_file.id]:
        file_info = openai_client.files.wait_for_processing(file_id)
        print(f"  File {file_id}: status={file_info.status}")
        if file_info.status != "processed":
            raise RuntimeError(f"File processing failed for {file_id}: {file_info}")
    print("Files ready!")


Reusing fine-tuning job: ftjob-26559512cf93499d9fd8eac439853f46


## 5. Submit Fine-Tuning Job

Now we submit the training job. SFT has three main hyperparameters — here's what each one does and why we chose these values:

- **`n_epochs`** (2): The number of complete passes through the training dataset. More epochs means the model sees each example more times, which can improve learning — but too many epochs leads to **overfitting** (the model memorizes the training data instead of learning general patterns). We use 2 because our experiments showed that 3 epochs caused the validation loss to rise 74% above its best value, a clear sign of overfitting on this dataset size.

- **`learning_rate_multiplier`** (0.8): Controls how aggressively the model updates its weights on each training step. Higher values mean faster learning but less stability; lower values are safer but may underutilize the data. We use 0.8 (slightly conservative) because higher values like 2.0 caused training instability in our experiments, while lower values like 0.5 produced weaker results.

- **`suffix`** ("bug-detection"): A name tag appended to your fine-tuned model ID. It doesn't affect training — it just makes the model easy to find later in your list of fine-tuned models.

With 224 examples and 2 epochs, training typically takes **15–30 minutes**.


In [6]:
ft_jobs = []


def submit_ft_job(method, training_type="GlobalStandard"):
    submitted_job = openai_client.fine_tuning.jobs.create(
        model=SFT_BASE_MODEL,
        training_file=train_file.id,
        validation_file=val_file.id,
        method=method,
        suffix="bug-detection",
        seed=105,
        extra_body={"trainingType": training_type},
    )
    ft_jobs.append(submitted_job.id)
    return submitted_job


if EXISTING_FINE_TUNE_JOB_ID:
    print("Loading existing fine-tuning job...")
    ft_job = openai_client.fine_tuning.jobs.retrieve(EXISTING_FINE_TUNE_JOB_ID)
    ft_jobs.append(ft_job.id)
else:
    print("Submitting fine-tuning job...")
    ft_job = submit_ft_job(
        {
            "type": "supervised",
            "supervised": {
                "hyperparameters": {
                    "n_epochs": 2,
                    "learning_rate_multiplier": 0.8,
                }
            },
        }
    )

print(f"Job ID: {ft_job.id}")
print(f"Status: {ft_job.status}")
print(f"Model: {ft_job.model}")


Loading existing fine-tuning job...


Job ID: ftjob-26559512cf93499d9fd8eac439853f46
Status: succeeded
Model: gpt-4.1-mini-2025-04-14


## 6. Monitor Training

The cell below polls the training job every 30 seconds and prints events as they arrive. The most important thing to watch is the **training loss** — it should decrease over time, meaning the model is learning from the data.

### What Healthy Training Looks Like

- ✅ **Healthy**: Loss decreases steadily over training steps. Validation loss tracks training loss (with a small gap).
- ⚠️ **Warning — overfitting**: Training loss keeps dropping but validation loss starts *increasing*. The model is memorizing rather than generalizing. Try fewer epochs or more training data.
- ⚠️ **Warning — instability**: Loss spikes or explodes in later steps. The learning rate may be too high — try a lower `learning_rate_multiplier`.
- ⚠️ **Warning — plateau**: Loss stops decreasing early. The model may need more data, more epochs, or a higher learning rate.

You can also monitor training in the [Azure AI Foundry portal](https://ai.azure.com) under **Build → Fine-tuning**, which provides a graphical loss curve.


In [7]:
def monitor_ft_job(job_id):
    print(f"Monitoring job {job_id}...\n")
    seen_events = set()
    while True:
        monitored_job = openai_client.fine_tuning.jobs.retrieve(job_id)
        events = openai_client.fine_tuning.jobs.list_events(job_id, limit=100)
        for event in reversed(events.data):
            if event.id not in seen_events:
                seen_events.add(event.id)
                print(f"  [{event.created_at}] {event.message}")
        print(f"  Status: {monitored_job.status}")
        if monitored_job.status in ("succeeded", "failed", "cancelled"):
            return monitored_job
        time.sleep(30)


job = monitor_ft_job(ft_job.id)
if job.status != "succeeded" and not EXISTING_FINE_TUNE_JOB_ID:
    print(
        f"\nInitial job {job.id} ended with status={job.status}; "
        "retrying on Standard tier with service-default learning rate."
    )
    retry_job = submit_ft_job(
        {
            "type": "supervised",
            "supervised": {"hyperparameters": {"n_epochs": 2}},
        },
        training_type="Standard",
    )
    print(f"Retry job ID: {retry_job.id}")
    job = monitor_ft_job(retry_job.id)

if job.status != "succeeded":
    raise RuntimeError(
        f"Fine-tuning did not succeed after {len(ft_jobs)} job(s); final status={job.status}"
    )

FINE_TUNED_MODEL = job.fine_tuned_model
print(f"\nFine-tuned model: {FINE_TUNED_MODEL}")


Monitoring job ftjob-26559512cf93499d9fd8eac439853f46...



  [1791064101] Job enqueued. Waiting for jobs ahead to complete.
  [1791064109] Preprocessing running for file training file.
  [1791064109] Preprocessing running for file validation file.
  [1791064110] Preprocessing completed for file training file.
  [1791064110] Preprocessing completed for file validation file.
  [1791064124] Job started.
  [1791064141] Data Import started.
  [1791064141] Finetuning started.
  [1791064600] Job queued, waiting for GPUs.
  [1791064777] Training started.
  [1791065289] Step 1: training loss=1.452401041984558
  [1791065310] Created results file: file-c625a150d2264ad48bd70d9a16a059ac
  [1791065336] Step 10: training loss=0.6727563738822937
  [1791065364] Step 20: training loss=0.4624280631542206
  [1791065389] Step 30: training loss=0.6521956324577332
  [1791065413] Step 40: training loss=0.25553256273269653
  [1791065436] Step 50: training loss=0.5418367385864258
  [1791065457] Step 60: training loss=0.4670926034450531
  [1791065476] Step 70: training 

## 7. Resolve the Fine-Tuned Deployment

The stable Azure AI Projects SDK currently exposes project deployment retrieval and listing. Set `FINE_TUNED_DEPLOYMENT` to the project deployment created for the succeeded fine-tuned model. This notebook resolves it through `AIProjectClient.deployments`, verifies the model match, and waits for inference readiness without using account endpoints or raw routes.


In [8]:
if not EXISTING_FINE_TUNED_DEPLOYMENT:
    raise ValueError(
        "FINE_TUNED_DEPLOYMENT must name the existing project deployment for the fine-tuned model"
    )

DEPLOYMENT_NAME = EXISTING_FINE_TUNED_DEPLOYMENT
project_deployment = project_client.deployments.get(DEPLOYMENT_NAME)
if project_deployment.model_name != FINE_TUNED_MODEL:
    raise RuntimeError(
        f"Deployment {DEPLOYMENT_NAME} targets {project_deployment.model_name}, "
        f"expected {FINE_TUNED_MODEL}"
    )

deployment_state = "available"
print(f"Project deployment resolved: {project_deployment.name}")
print(f"Deployment model verified: {project_deployment.model_name}")


Project deployment resolved: bug-detection-ft-39853f46
Deployment model verified: gpt-4.1-mini-2025-04-14.ft-26559512cf93499d9fd8eac439853f46-bug-detection


In [9]:
# Poll until the fine-tuned deployment accepts inference traffic.
for attempt in range(1, 21):
    try:
        project_deployment = project_client.deployments.get(DEPLOYMENT_NAME)
        print(f"Warm-up check {attempt}/20: deployment listed as {project_deployment.name}")
        get_completion(
            DEPLOYMENT_NAME,
            [{"role": "user", "content": "Reply exactly READY"}],
            64,
        )
        print("Fine-tuned deployment is ready.")
        deployment_state = "ready"
        break
    except Exception as exc:
        print(f"  Warm-up check failed: {type(exc).__name__}: {exc}")
    if attempt == 20:
        raise RuntimeError("Fine-tuned deployment did not become ready")
    time.sleep(30)


Warm-up check 1/20: deployment listed as bug-detection-ft-39853f46

Fine-tuned deployment is ready.


## 8. Evaluate the Fine-Tuned Model

The key question: **did fine-tuning help?**

We run the exact same test examples from Section 3 through the fine-tuned model, scored by the same judge with the same rubric. This apples-to-apples comparison tells us whether the training data actually improved the model's bug detection ability.

If the fine-tuned model's pass rate and average score are higher than the baseline, we know SFT worked. If they're the same or worse, we'd need to revisit our training data quality, hyperparameters, or dataset size.


In [10]:
print("Evaluating fine-tuned model...\n")
ft_eval = evaluate_model(DEPLOYMENT_NAME, test_examples, label="Fine-Tuned")

print("\n--- Baseline vs Fine-Tuned ---")
print(f"  {'Metric':<25} {'Baseline':>10} {'Fine-Tuned':>10}")
print(f"  {'-'*25} {'-'*10} {'-'*10}")
print(f"  {'Combined Score':<25} {baseline_eval['avg_combined']:>10.2f} {ft_eval['avg_combined']:>10.2f}")
print(f"  {'Pass Rate (>=7.0)':<25} {baseline_eval['pass_rate']:>9.1f}% {ft_eval['pass_rate']:>9.1f}%")

Evaluating fine-tuned model...



  [Fine-Tuned] Example 1: bug=5 expl=4 fix=2 combined=3.7


  [Fine-Tuned] Example 2: bug=10 expl=9 fix=10 combined=9.7


  [Fine-Tuned] Example 3: bug=10 expl=10 fix=9 combined=9.7


  [Fine-Tuned] Example 4: bug=10 expl=8 fix=9 combined=9.0


...

  [Fine-Tuned] Example 6: bug=10 expl=10 fix=10 combined=10.0


  [Fine-Tuned] Example 7: bug=10 expl=9 fix=10 combined=9.7


  [Fine-Tuned] Example 8: bug=10 expl=10 fix=10 combined=10.0


  [Fine-Tuned] Example 9: bug=10 expl=7 fix=10 combined=9.0


  [Fine-Tuned] Example 10: bug=10 expl=9 fix=8 combined=9.0

  Fine-Tuned Summary:
    Average combined score: 8.97
    Pass rate (>=7.0): 9/10 (90.0%)

--- Baseline vs Fine-Tuned ---
  Metric                      Baseline Fine-Tuned
  ------------------------- ---------- ----------
  Combined Score                  8.57       8.97
  Pass Rate (>=7.0)              90.0%      90.0%


## 9. Compare to the Teacher Model

Our final comparison: how does the fine-tuned mini model stack up against **GPT-5.4** — the teacher that generated the training data?

This is the payoff of distillation. If the student matches or approaches the teacher's quality, we've successfully transferred that capability at **much lower cost and latency**. And if the student *beats* the teacher on some metrics (which happens more often than you'd expect), it's because fine-tuning teaches a consistent, focused output style that the larger model doesn't naturally produce.

We evaluate GPT-5.4 on the same test set with the same judge, then display all three models side by side — including cost and latency comparisons.


In [11]:
print("Evaluating teacher model (GPT-5.4)...\n")
teacher_eval = evaluate_model(JUDGE_MODEL, test_examples, label="Teacher")

Evaluating teacher model (GPT-5.4)...



  [Teacher] Example 1: bug=5 expl=6 fix=3 combined=4.7


  [Teacher] Example 2: bug=10 expl=10 fix=10 combined=10.0


  [Teacher] Example 3: bug=10 expl=10 fix=8 combined=9.3


  [Teacher] Example 4: bug=10 expl=10 fix=9 combined=9.7


...

  [Teacher] Example 6: bug=10 expl=10 fix=10 combined=10.0


  [Teacher] Example 7: bug=10 expl=10 fix=10 combined=10.0


  [Teacher] Example 8: bug=10 expl=10 fix=10 combined=10.0


  [Teacher] Example 9: bug=10 expl=10 fix=6 combined=8.7


  [Teacher] Example 10: bug=10 expl=10 fix=10 combined=10.0

  Teacher Summary:
    Average combined score: 9.23
    Pass rate (>=7.0): 9/10 (90.0%)


In [12]:
# Final comparison table
print("\n" + "=" * 110)
print("FINAL COMPARISON")
print("=" * 110)

# Verified pricing (per 1M tokens) and TTFT from artificialanalysis.ai
model_info = {
    "gpt-4.1-mini (base)": {
        "combined": baseline_eval["avg_combined"],
        "pass_rate": baseline_eval["pass_rate"],
        "input_cost": 0.40,
        "output_cost": 1.60,
        "ttft": 1.35,
    },
    "gpt-4.1-mini FT": {
        "combined": ft_eval["avg_combined"],
        "pass_rate": ft_eval["pass_rate"],
        "input_cost": 0.40,
        "output_cost": 1.60,
        "ttft": 1.35,
    },
    "gpt-5.4 (teacher)": {
        "combined": teacher_eval["avg_combined"],
        "pass_rate": teacher_eval["pass_rate"],
        "input_cost": 2.50,
        "output_cost": 15.00,
        "ttft": 181.2,
    },
}

teacher = model_info["gpt-5.4 (teacher)"]

header = (
    f"{'Model':<22} {'Combined':>8} {'Pass@7':>7} "
    f"{'In $/1M':>8} {'Out $/1M':>9} {'TTFT(s)':>8} "
    f"{'vs Teacher Q':>13} {'vs Teacher $':>13} {'vs Teacher Lat':>15}"
)
print(header)
print("-" * 110)

for name, info in model_info.items():
    # Compute vs-teacher metrics
    q_ratio = info["combined"] / teacher["combined"] * 100 if teacher["combined"] > 0 else 0
    avg_cost = (info["input_cost"] + info["output_cost"]) / 2
    teacher_avg_cost = (teacher["input_cost"] + teacher["output_cost"]) / 2
    cost_ratio = avg_cost / teacher_avg_cost * 100 if teacher_avg_cost > 0 else 0
    lat_ratio = info["ttft"] / teacher["ttft"] * 100 if teacher["ttft"] > 0 else 0

    row = (
        f"{name:<22} {info['combined']:>8.2f} {info['pass_rate']:>6.1f}% "
        f"${info['input_cost']:>6.2f} ${info['output_cost']:>7.2f} {info['ttft']:>8.2f} "
        f"{q_ratio:>12.1f}% {cost_ratio:>12.1f}% {lat_ratio:>14.1f}%"
    )
    print(row)

print("=" * 110)
print("\nKey takeaway: The fine-tuned mini model delivers teacher-level quality")
print("at a fraction of the cost and latency.")

execution_summary = {
    "project_endpoint": FOUNDRY_PROJECT_ENDPOINT,
    "base_deployment": BASE_MODEL,
    "judge_deployment": JUDGE_MODEL,
    "sft_base_model": SFT_BASE_MODEL,
    "fine_tuning_job_ids": ft_jobs,
    "successful_job_id": job.id,
    "fine_tuned_model": FINE_TUNED_MODEL,
    "fine_tuned_deployment": DEPLOYMENT_NAME,
    "fine_tuning_status": job.status,
    "deployment_status": deployment_state,
    "evaluation_status": "completed",
    "baseline": {
        "combined_score": baseline_eval["avg_combined"],
        "pass_rate": baseline_eval["pass_rate"],
    },
    "fine_tuned": {
        "combined_score": ft_eval["avg_combined"],
        "pass_rate": ft_eval["pass_rate"],
    },
    "teacher": {
        "combined_score": teacher_eval["avg_combined"],
        "pass_rate": teacher_eval["pass_rate"],
    },
}
with open("execution_summary.json", "w", encoding="utf-8") as summary_file:
    json.dump(execution_summary, summary_file, indent=2)
print("Evaluation status: completed")
print("Saved execution_summary.json")



FINAL COMPARISON
Model                  Combined  Pass@7  In $/1M  Out $/1M  TTFT(s)  vs Teacher Q  vs Teacher $  vs Teacher Lat
--------------------------------------------------------------------------------------------------------------
gpt-4.1-mini (base)        8.57   90.0% $  0.40 $   1.60     1.35         92.8%         11.4%            0.7%
gpt-4.1-mini FT            8.97   90.0% $  0.40 $   1.60     1.35         97.1%         11.4%            0.7%
gpt-5.4 (teacher)          9.23   90.0% $  2.50 $  15.00   181.20        100.0%        100.0%          100.0%

Key takeaway: The fine-tuned mini model delivers teacher-level quality
at a fraction of the cost and latency.
Evaluation status: completed
Saved execution_summary.json


## 10. Close Clients

The existing project deployment is retained. Remove it through your normal project deployment workflow when it is no longer needed.


In [13]:
openai_client.close()
project_client.close()
credential.close()
print("\nDone! Clients closed; project deployment retained.")



Done! Clients closed; project deployment retained.
